# Challenge 1 - T-test

In statistics, t-test is used to test if two data samples have a significant difference between their means. There are two types of t-test:

* **Student's t-test** (a.k.a. independent or uncorrelated t-test). This type of t-test is to compare the samples of **two independent populations** (e.g. test scores of students in two different classes). `scipy` provides the [`ttest_ind`](https://docs.scipy.org/doc/scipy-0.15.1/reference/generated/scipy.stats.ttest_ind.html) method to conduct student's t-test.

* **Paired t-test** (a.k.a. dependent or correlated t-test). This type of t-test is to compare the samples of **the same population** (e.g. scores of different tests of students in the same class). `scipy` provides the [`ttest_re`](https://docs.scipy.org/doc/scipy-0.15.1/reference/generated/scipy.stats.ttest_rel.html) method to conduct paired t-test.

Both types of t-tests return a number which is called the **p-value**. If p-value is below 0.05, we can confidently declare the null-hypothesis is rejected and the difference is significant. If p-value is between 0.05 and 0.1, we may also declare the null-hypothesis is rejected but we are not highly confident. If p-value is above 0.1 we do not reject the null-hypothesis.

Read more about the t-test in [this article](https://researchbasics.education.uconn.edu/t-test/) and [this Quora](https://www.quora.com/What-is-the-difference-between-a-paired-and-unpaired-t-test). Make sure you understand when to use which type of t-test. 

In [29]:
# Import libraries

import pandas as pd
from scipy import stats

#### Import dataset

In this challenge we will work on the Pokemon dataset you have used last week. The goal is to test whether different groups of pokemon (e.g. Legendary vs Normal, Generation 1 vs 2, single-type vs dual-type) have different stats (e.g. HP, Attack, Defense, etc.).

In [30]:
# Import dataset

pokemon = pd.read_csv("Pokemon.csv")

pokemon.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


#### First we want to define a function with which we can test the means of a feature set of two samples. 

In the next cell you'll see the annotations of the Python function that explains what this function does and its arguments and returned value. This type of annotation is called **docstring** which is a convention used among Python developers. The docstring convention allows developers to write consistent tech documentations for their codes so that others can read. It also allows some websites to automatically parse the docstrings and display user-friendly documentations.

Follow the specifications of the docstring and complete the function.

In [31]:
def t_test_features(
    s1, s2, features=["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", "Total"]
):
    """Test means of a feature set of two samples

    Args:
        s1 (dataframe): sample 1
        s2 (dataframe): sample 2
        features (list): an array of features to test

    Returns:
        dict: a dictionary of t-test scores for each feature where the feature name is the key and the p-value is the value
    """
    results = {}

    for feature in features:
        # Perform an independent t-test
        # equal_var=False is often safer if we don't assume equal variance (Welch's T-test)
        t_stat, p_val = stats.ttest_ind(s1[feature], s2[feature], equal_var=False)

        # Store the p-value in the results dictionary
        results[feature] = p_val

    return results

#### Using the `t_test_features` function, conduct t-test for Lengendary vs non-Legendary pokemons.

*Hint: your output should look like below:*

```
{'HP': 1.0026911708035284e-13,
 'Attack': 2.520372449236646e-16,
 'Defense': 4.8269984949193316e-11,
 'Sp. Atk': 1.5514614112239812e-21,
 'Sp. Def': 2.2949327864052826e-15,
 'Speed': 1.049016311882451e-18,
 'Total': 9.357954335957446e-47}
 ```

In [32]:
# Your code here
# 1. Create the two samples
legendary_df = pokemon[pokemon["Legendary"] == True]
non_legendary_df = pokemon[pokemon["Legendary"] == False]

# 2. Run the t-test function
pokemon_stats_comparison = t_test_features(legendary_df, non_legendary_df)

# 3. Display the results clearly

results_df = pd.DataFrame.from_dict(pokemon_stats_comparison, orient="index", columns=["p-value"])
print(results_df)

              p-value
HP       1.002691e-13
Attack   2.520372e-16
Defense  4.826998e-11
Sp. Atk  1.551461e-21
Sp. Def  2.294933e-15
Speed    1.049016e-18
Total    9.357954e-47


#### From the test results above, what conclusion can you make? Do Legendary and non-Legendary pokemons have significantly different stats on each feature?

In [33]:
# Your comment here
# Every single P-value is significantly lower that the standard treshold of 0.05, there is practically zero chance that the difference in power betweem Legendary and Non-Legendary pokemon
# is accidental. Legendary Pokemon are statistically superior across every single metric.

#### Next, conduct t-test for Generation 1 and Generation 2 pokemons.

In [34]:
# Your code here
# 1) create the two samples
first_generation = pokemon[pokemon["Generation"] == 1]
second_generation = pokemon[pokemon["Generation"] == 2]

# 2) Run the T-test
pokemon_stats_comparison = t_test_features(first_generation, second_generation)

# 3. Display the results clearly
results_df = pd.DataFrame.from_dict(
    pokemon_stats_comparison, orient="index", columns=["p-value"]
).round(4)
print(results_df)

         p-value
HP        0.1455
Attack    0.2472
Defense   0.5678
Sp. Atk   0.1233
Sp. Def   0.1883
Speed     0.0024
Total     0.5631


#### What conclusions can you make?

In [35]:
# Your comment here
# The speed is the only significat difference, Defense 0.5678  - this is very high so when it comes to defence both generations are indetical

#### Compare pokemons who have single type vs those having two types.

In [36]:
# Your code her
# Creating the two samples
# Create a boolean column: True if the Pokémon has two types
pokemon["is_dual_type"] = pokemon["Type 2"].notna()

# 2) Run the T-test
pokemon_stats_comparison = t_test_features(
    pokemon[pokemon["is_dual_type"] == True], pokemon[pokemon["is_dual_type"] == False]
)

# 3. Display the results clearly
results_df = pd.DataFrame.from_dict(
    pokemon_stats_comparison, orient="index", columns=["p-value"]
).round(4)
print(results_df)

         p-value
HP        0.1131
Attack    0.0001
Defense   0.0000
Sp. Atk   0.0001
Sp. Def   0.0001
Speed     0.0242
Total     0.0000


#### What conclusions can you make?

In [37]:
# Your comment here
# Except for HP, every single attribute shows a statistically significant difference.

#### Now, we want to compare whether there are significant differences of `Attack` vs `Defense`  and  `Sp. Atk` vs `Sp. Def` of all pokemons. Please write your code below.

*Hint: are you comparing different populations or the same population?*

In [38]:
# Your code here
# 1. Conduct Paired T-test for Attack vs Defense
t_phys, p_phys = stats.ttest_rel(pokemon['Attack'], pokemon['Defense'])

# 2. Conduct Paired T-test for Sp. Atk vs Sp. Def
t_spec, p_spec = stats.ttest_rel(pokemon['Sp. Atk'], pokemon['Sp. Def'])

# 3. Create a clean results summary
comparison_results = pd.DataFrame({
    'Comparison': ['Attack vs Defense', 'Sp. Atk vs Sp. Def'],
    't-statistic': [t_phys, t_spec],
    'p-value': [p_phys, p_spec]
}).round(4)

print(comparison_results)

           Comparison  t-statistic  p-value
0   Attack vs Defense       4.3256   0.0000
1  Sp. Atk vs Sp. Def       0.8540   0.3934


#### What conclusions can you make?

In [ ]:
# Your comment here
''' 
Attack vs. Defense:
p-value (0.0000) This is highly significant. It confirms that there is a systematic difference between Attack and Defense across the Pokémon universe.
t-statistic (4.3256): Since this is positive, it means that Attack is significantly higher than Defense on average.
The Interpretation: Pokémon are designed to be "glass cannons" or attackers more often than they are designed to be "tanks." In the physical category, 
the "sword" is statistically sharper than the "shield." 
'''

'''
Sp. Atk vs. Sp. Def:
p-value (0.3934): This is much larger than 0.05.
The Interpretation: This result is not significant. It tells us that, on average, Special Attack and Special Defense are basically equal.
Analyst Insight: While physical combat is biased toward the attacker, special combat is much more balanced. If you find a Pokémon with a high Special Attack, 
there is a statistically better chance there's a "Special Wall" out there that can match it, compared to physical combat.

'''

